# Integrales-03 — La bulle et la première divergence

Le carnet 01 calculait le triangle à une boucle : en $d = 4$, son comptage de
puissances vaut $4 - 2 \times 3 = -2$, et tout y restait fini. Le carnet 02
poussait au sunrise et voyait les polylogarithmes sortir — même comptage,
encore $-2$, cette fois en $d = 2$. Celui-ci descend d'un étage — **deux
propagateurs, la bulle** — et le comptage passe à $4 - 2 \times 2 = 0$, la
première fois que la série n'est plus protégée :

$$B(\Lambda) = \int_{|k| < \Lambda} \frac{d^4 k}{k^2 (k-p)^2}, \qquad p^2 = 1 .$$

C'est le plus petit graphe dont l'intégrale **diverge** : l'intégrande décroît
comme $1/k^4$ en $|k| \to \infty$, exactement la dimension du volume —
l'intégrale croît comme $\ln \Lambda^2$. Le programme du carnet :

1. **la mesure de la sphère $S^3$** — vérifiée numériquement avant tout usage
   (une première écriture de ce carnet utilisait $2\pi\sin^2\psi\, d\psi$ et
   rendait la moitié de l'intégrale : le contrôle importe) ;
2. **une forme fermée** par paramètre de Feynman, avec son domaine de validité
   exhibé (la boule dure ne commute pas au décalage — l'écart est mesuré, pas
   masqué) ;
3. **la divergence mesurée** : coefficient de $\ln \Lambda^2$ et constante
   finie, chacune ancrée dans une *planted truth* indépendante ;
4. **la régularisation soustractive** : un paramètre $\mu$, une limite finie.

Comme dans les carnets 01 et 02 : outil réel (BCL .NET, `BigInteger` à échelle fixe et quadrature en `double`), chaque
affirmation quantitative vit dans une sortie commise, les refus sont documentés.

## 1. La mesure de $S^3$

En quatre dimensions, $d^4 k = k^3\,dk\,d\Omega_3$ et la sphère unité se
paramètre par l'angle $\psi$ entre $k$ et $p$ :

$$d\Omega_3 = \sin^2\psi\, d\psi\, d\Omega_2, \qquad \int d\Omega_2 = 4\pi
\quad\Longrightarrow\quad \int_{S^3} d\Omega_3 = 4\pi \int_0^\pi \sin^2\psi\, d\psi = 2\pi^2 .$$

La cellule suivante vérifie l'aire par quadrature — c'est la *planted truth* de
la mesure : toute la suite en dépend.

In [1]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

var p2 = BigInteger.One;  // p^2 = 1 : l'echelle
var culture = CultureInfo.InvariantCulture;
string Inv(FormattableString texte) => texte.ToString(culture);
string Verdict(bool conforme) => conforme ? "CONFORME" : "ÉCART";
void Verifier(bool condition, string message)
{
    if (!condition) throw new InvalidOperationException(message);
}

// Seuil absolu des quadratures stabilisées : au-dessus du bruit d'arrondi
// du double pour ces intégrales, mais très en dessous de l'effet du coin.
const double TolQuad = 1e-10;
const int DemiMaillage = 256;
const int DemiMaillageFin = 2 * DemiMaillage;
const double BorneTanhSinh = 4.0;
var grillesQuad = new Dictionary<int, (double Gauche, double Droite, double Poids)[]>();

// Tanh–sinh sur u ∈ [-4,4], h = 4/m, 2m+1 nœuds par intervalle.
// On conserve séparément les deux distances normalisées aux bords :
// calculer la petite distance par 1 - grandeDistance la ferait disparaître.
(double Gauche, double Droite, double Poids)[] GrilleQuad(int m)
{
    if (m <= 0) throw new ArgumentOutOfRangeException(nameof(m));
    if (grillesQuad.TryGetValue(m, out var grille)) return grille;
    double h = BorneTanhSinh / m;
    grille = new (double Gauche, double Droite, double Poids)[2 * m + 1];
    for (int j = -m; j <= m; j++)
    {
        double u = Math.Abs(j * h);
        double z = Math.Exp(-Math.PI * Math.Sinh(u));
        double den = 1.0 + z;
        double petit = z / den, grand = 1.0 / den;
        double poids = h * Math.PI * Math.Cosh(u) * z / (den * den);
        if (Math.Abs(j) == m) poids *= 0.5; // extrémités de la règle trapézoïdale en u
        grille[j + m] = j < 0 ? (petit, grand, poids) : (grand, petit, poids);
    }
    grillesQuad.Add(m, grille);
    return grille;
}

// f reçoit x-a et b-x sur [a,b] ; la somme pondérée est compensée.
double Quad(Func<double, double, double> f, double a, double b, int m)
{
    if (a == b) return 0.0;
    if (a > b) throw new ArgumentOutOfRangeException(nameof(b));
    double longueur = b - a, somme = 0.0, compensation = 0.0;
    foreach (var (gauche, droite, poids) in GrilleQuad(m))
    {
        double terme = poids * f(longueur * gauche, longueur * droite) - compensation;
        double suivant = somme + terme;
        compensation = (suivant - somme) - terme;
        somme = suivant;
    }
    return longueur * somme;
}

// Comparaison en Decimal du texte G17 du double avec la citation décimale.
// Cela évite de convertir d'abord la référence entière en un double identique.
decimal EcartCitation(double calcule, string cite) => Math.Abs(
    decimal.Parse(calcule.ToString("G17", culture), NumberStyles.Float, culture)
    - decimal.Parse(cite, NumberStyles.Float, culture));

Console.WriteLine(Inv($"Tanh–sinh double, u ∈ [-{BorneTanhSinh:G}, {BorneTanhSinh:G}] : h = {BorneTanhSinh / DemiMaillage:G} → {BorneTanhSinh / DemiMaillageFin:G}, {2 * DemiMaillage + 1} → {2 * DemiMaillageFin + 1} nœuds par intervalle ; seuil absolu = {TolQuad:G}."));

// aire de S^3 par quadrature (planted truth de la mesure)
const string RefAire = "19.7392088021787172376689819998"; // Python, cellule 2 : « aire S^3 mesuree » et « 2 pi^2 »
const string RefAireRel = "0.0"; // Python, cellule 2 : « ecart relatif »
double MesureAire(double psi, double reste)
{
    double sinus = Math.Sin(Math.Min(psi, reste)); // sin(π − ψ) = sin ψ
    return 4.0 * Math.PI * sinus * sinus;
}
double aireN = Quad(MesureAire, 0.0, Math.PI, DemiMaillage);
double aire = Quad(MesureAire, 0.0, Math.PI, DemiMaillageFin);
double aireTheo = 2.0 * Math.PI * Math.PI;
double aireRel = Math.Abs(aire - aireTheo) / aireTheo;
double aireRaff = Math.Abs(aire - aireN);
decimal aireRef = EcartCitation(aire, RefAire);
decimal aireTheoRef = EcartCitation(aireTheo, RefAire);
decimal aireRelRef = EcartCitation(aireRel, RefAireRel);
bool aireOk = aireRaff <= TolQuad && Math.Abs(aire - aireTheo) <= TolQuad
    && aireRef <= (decimal)TolQuad && aireTheoRef <= (decimal)TolQuad
    && aireRelRef <= (decimal)TolQuad / (decimal)aireTheo;
Console.WriteLine(Inv($"aire S^3 mesurée = {aire:G17} ; Python, cellule 2, cité = {RefAire} ; écart = {aireRef:E6}"));
Console.WriteLine(Inv($"2 pi^2           = {aireTheo:G17} ; Python, cellule 2, cité = {RefAire} ; écart = {aireTheoRef:E6}"));
Console.WriteLine(Inv($"écart relatif    = {aireRel:G17} ; Python, cellule 2, cité = {RefAireRel} ; écart = {aireRelRef:E6}"));
Console.WriteLine(Inv($"Raffinement : {aireN:G17} → {aire:G17} ; |Δ| = {aireRaff:E6} ; seuil = {TolQuad:G} : {Verdict(aireOk)}"));
Verifier(aireOk, "Le contrôle de l'aire de S^3 a échoué.");

Tanh–sinh double, u ∈ [-4, 4] : h = 0.015625 → 0.0078125, 513 → 1025 nœuds par intervalle ; seuil absolu = 1E-10.


aire S^3 mesurée = 19.739208802178716 ; Python, cellule 2, cité = 19.7392088021787172376689819998 ; écart = 1.237669E-015


2 pi^2           = 19.739208802178716 ; Python, cellule 2, cité = 19.7392088021787172376689819998 ; écart = 1.237669E-015


écart relatif    = 0 ; Python, cellule 2, cité = 0.0 ; écart = 0.000000E+000


Raffinement : 19.739208802178716 → 19.739208802178716 ; |Δ| = 0.000000E+000 ; seuil = 1E-10 : CONFORME


## 2. La forme fermée : paramètre de Feynman et complétion du carré

L'identité $\frac{1}{AB} = \int_0^1 \frac{dx}{(xA + (1-x)B)^2}$ avec
$A = k^2$, $B = (k-p)^2$ donne, après complétion du carré,

$$\frac{1}{k^2 (k-p)^2} = \int_0^1 \frac{dx}{\big((k - (1-x)p)^2 + \Delta\big)^2},
\qquad \Delta = x(1-x)\, p^2 .$$

**Avertissement honnête** : la forme fermée ci-dessous échange
$\int_{|k|<\Lambda}$ contre $\int_{|q|<\Lambda}$ après le décalage
$q = k - (1-x)p$. Ce geste est exact en régularisation dimensionnelle (où la
translation est une invariance), mais une **boule dure ne commute pas au
décalage** : l'écart est d'ordre $\bar\Delta/\Lambda^2$, avec
$\bar\Delta = p^2/6 = \int_0^1 x(1-x)p^2\,dx$ — mesuré en section 3, son
coefficient relevé en section 8, pas masqué. L'intégrale radiale, elle, est
exacte :

$$\int_0^\Lambda \frac{k^3\,dk}{(k^2+\Delta)^2} = \frac{1}{2}\left[
\ln\frac{\Lambda^2+\Delta}{\Delta} + \frac{\Delta}{\Lambda^2+\Delta} - 1\right].$$

In [2]:
// Forme fermée : calcul à échelle fixe et contrôle de l’intégrande de Feynman
// Les rationnels sont des paires d'entiers exacts jusqu'à leur projection
// sur l'échelle 10^60. Les divisions BigInteger tronquent vers zéro.
const int DecimalesFixes = 60;
const int NbTermesCorrection = 112;
var echelleFixe = BigInteger.Pow(10, DecimalesFixes);
BigInteger MultFixe(BigInteger a, BigInteger b) => a * b / echelleFixe;
BigInteger DivFixe(BigInteger a, BigInteger b) => a * echelleFixe / b;
double DoubleFixe(BigInteger a) => (double)a / (double)echelleFixe;
string TexteFixe(BigInteger a)
{
    string chiffres = BigInteger.Abs(a).ToString(culture).PadLeft(DecimalesFixes + 1, '0');
    string entier = chiffres.Substring(0, chiffres.Length - DecimalesFixes);
    string fraction = chiffres.Substring(chiffres.Length - DecimalesFixes).TrimEnd('0');
    return (a.Sign < 0 ? "-" : "") + entier + (fraction.Length == 0 ? "" : "." + fraction);
}
BigInteger LireFixe(string texte)
{
    bool negatif = texte.StartsWith("-", StringComparison.Ordinal);
    var champs = (negatif ? texte.Substring(1) : texte).Split('.');
    string fraction = champs.Length == 2 ? champs[1] : "";
    if (champs.Length > 2 || fraction.Length > DecimalesFixes) throw new FormatException(texte);
    var entier = BigInteger.Parse(champs[0] + fraction.PadRight(DecimalesFixes, '0'), culture);
    return negatif ? -entier : entier;
}

// π par Machin : 16 arctan(1/5) − 4 arctan(1/239).
// La première fraction projetée à zéro borne le reste de la série alternée.
BigInteger AtanInverseFixe(int q)
{
    BigInteger somme = 0, puissance = q, carre = (BigInteger)q * q;
    for (int n = 0; ; n++, puissance *= carre)
    {
        var terme = echelleFixe / ((2 * n + 1) * puissance);
        if (terme.IsZero) return somme;
        somme += n % 2 == 0 ? terme : -terme;
    }
}
var piFixe = 16 * AtanInverseFixe(5) - 4 * AtanInverseFixe(239);
var pi2Fixe = MultFixe(piFixe, piFixe);

// ln(y) = 2 atanh((y−1)/(y+1)) ; la réduction par puissances de 2
// donne |z| ≤ 1/3, ce qui borne la queue géométrique dès le premier
// terme sous l'unité d'échelle. Les puissances de z restent rationnelles.
BigInteger LogAtanhFixe(BigInteger n, BigInteger d)
{
    BigInteger u = n - d, v = n + d, un = u, vn = v, u2 = u * u, v2 = v * v;
    BigInteger somme = 0;
    for (int i = 0; ; i++, un *= u2, vn *= v2)
    {
        var terme = 2 * echelleFixe * un / ((2 * i + 1) * vn);
        if (terme.IsZero) return somme;
        somme += terme;
    }
}
var ln2Fixe = LogAtanhFixe(2, 1);
BigInteger LogFixe(BigInteger n, BigInteger d)
{
    if (n.Sign <= 0 || d.Sign <= 0) throw new ArgumentOutOfRangeException(nameof(n));
    int k = 0;
    while (n < d) { n *= 2; k--; }
    while (n >= 2 * d) { d *= 2; k++; }
    return k * ln2Fixe + LogAtanhFixe(n, d);
}

// Avec z = Δ/Λ², le crochet est ln(Λ²/p²) − ln(x(1−x)) − 1
// + ln(1+z) + z/(1+z). L'intégrale de −ln(x(1−x)) vaut 2.
// La correction développe ln(1+z) + z/(1+z) en termes
// (−1)^(n+1) (n+1)/n × z^n. Le moment de [x(1−x)]^n vaut
// (n!)²/(2n+1)! ; son numérateur et son dénominateur sont exacts.
BigInteger CorrectionCrochetFixe(BigInteger lam, BigInteger pCarre)
{
    var l2 = lam * lam;
    if (lam.Sign <= 0 || pCarre.Sign <= 0 || pCarre > l2)
        throw new ArgumentOutOfRangeException(nameof(pCarre), "Il faut 0 < p² ≤ Λ² et Λ > 0.");
    // Ici 0 ≤ z ≤ 1/4 : la queue est majorée par 2(1/4)^113/(1−1/4).
    BigInteger somme = 0, betaNum = 1, betaDen = 1, qNum = 1, qDen = 1;
    for (int n = 1; n <= NbTermesCorrection; n++)
    {
        betaNum *= n * n;
        betaDen *= (2 * n) * (2 * n + 1);
        qNum *= pCarre;
        qDen *= l2;
        var terme = echelleFixe * (n + 1) * betaNum * qNum / (n * betaDen * qDen);
        if (terme.IsZero) break; // les termes restants, décroissants, sont aussi sous l'échelle
        somme += n % 2 == 1 ? terme : -terme;
    }
    return somme;
}
BigInteger Residufixe(BigInteger lam, BigInteger pCarre) => MultFixe(pi2Fixe, CorrectionCrochetFixe(lam, pCarre));
BigInteger BFerme(BigInteger lam, BigInteger pCarre) =>
    MultFixe(pi2Fixe, LogFixe(lam * lam, pCarre) + echelleFixe + CorrectionCrochetFixe(lam, pCarre)); // 2 pi^2 * 1/2 = pi^2 devant le crochet

// Contrôle indépendant : même crochet de Feynman que dans la source,
// quadraturé directement, sans utiliser les moments analytiques.
double BFermeQuad(double lam, double pCarre, int m) => Math.PI * Math.PI * Quad((x, unMoinsX) =>
{
    double delta = x * unMoinsX * pCarre, a = lam * lam + delta;
    return Math.Log(a / delta) + delta / a - 1.0;
}, 0.0, 1.0, m);

// Python, cellule 4 : lignes imprimées « B_ferme(Lam=2) » et « B_ferme(Lam=8) ».
// Tolérances de comparaison à échelle fixe : une unité du dernier chiffre cité.
var casFermes = new[] {
    (Lam: 2, Ref: "24.34480199630533702092892284", Tol: "0.00000000000000000000000001"),
    (Lam: 8, Ref: "50.9674192526556407971513075656", Tol: "0.0000000000000000000000000001")
};
Console.WriteLine(Inv($"Forme fermée : échelle 10^{DecimalesFixes}, troncature vers zéro ; au plus {NbTermesCorrection} termes de correction ; seuil quadrature = {TolQuad:G}."));
var ecartsFermes = new List<(int Lam, BigInteger EcartFixe, decimal EcartQuad, double Raffinement)>();
foreach (var cas in casFermes)
{
    var fixe = BFerme(cas.Lam, p2);
    var ecartFixe = BigInteger.Abs(fixe - LireFixe(cas.Ref));
    double qN = BFermeQuad(cas.Lam, (double)p2, DemiMaillage);
    double qFin = BFermeQuad(cas.Lam, (double)p2, DemiMaillageFin);
    double raff = Math.Abs(qFin - qN), ecartInterne = Math.Abs(qFin - DoubleFixe(fixe));
    decimal ecartQuad = EcartCitation(qFin, cas.Ref);
    bool okFixe = ecartFixe <= LireFixe(cas.Tol);
    bool okQuad = raff <= TolQuad && ecartInterne <= TolQuad && ecartQuad <= (decimal)TolQuad;
    ecartsFermes.Add((cas.Lam, ecartFixe, ecartQuad, raff));
    Console.WriteLine(Inv($"B_ferme(Lam={cas.Lam}) = {TexteFixe(fixe)} ; Python, cellule 4, cité = {cas.Ref}"));
    Console.WriteLine(Inv($"  Écart à échelle fixe = {DoubleFixe(ecartFixe):E6} ; seuil = {cas.Tol} : {Verdict(okFixe)}"));
    Console.WriteLine(Inv($"  Quadrature : {qN:G17} → {qFin:G17} ; |Δ| = {raff:E6} ; écart à la citation = {ecartQuad:E6} ; écart à la forme fermée = {ecartInterne:E6} : {Verdict(okQuad)}"));
    Verifier(okFixe && okQuad, Inv($"Échec du contrôle de la forme fermée pour Λ = {cas.Lam}."));
}

Forme fermée : échelle 10^60, troncature vers zéro ; au plus 112 termes de correction ; seuil quadrature = 1E-10.


B_ferme(Lam=2) = 24.344801996305337020928922839975493758618268670964211765442133 ; Python, cellule 4, cité = 24.34480199630533702092892284


  Écart à échelle fixe = 2.450624E-029 ; seuil = 0.00000000000000000000000001 : CONFORME


  Quadrature : 24.344801996305335 → 24.344801996305335 ; |Δ| = 0.000000E+000 ; écart à la citation = 2.020929E-015 ; écart à la forme fermée = 0.000000E+000 : CONFORME


B_ferme(Lam=8) = 50.967419252655640797151307565586231389532341197890495199244284 ; Python, cellule 4, cité = 50.9674192526556407971513075656


  Écart à échelle fixe = 1.376861E-029 ; seuil = 0.0000000000000000000000000001 : CONFORME


  Quadrature : 50.967419252655638 → 50.967419252655638 ; |Δ| = 0.000000E+000 ; écart à la citation = 2.797151E-015 ; écart à la forme fermée = 7.105427E-015 : CONFORME


## 3. La voie directe : quadrature sur la boule, et le facteur 2 attrapé en direct

La voie indépendante n'utilise **aucun** paramètre de Feynman : elle intègre
l'intégrande d'origine sur $\psi \in [0, \pi]$ puis $k \in [0, \Lambda]$.
La première version de ce carnet écrivait la mesure angulaire
$2\pi\sin^2\psi$ — soit la moitié de $4\pi\sin^2\psi$ — et rendait
exactement la moitié de l'intégrale. Le contrôle d'aire de la section 1 est ce
qui tranche.

**Ce que cette première version ne voyait pas.** Une fois l'angle intégré,
l'intégrande radial vaut exactement $\dfrac{2\pi^2 k}{\max(k^2, p^2)}$ : il est
**continu mais anguleux en $k = p$**, et les points de rupture de la quadrature
($[0,\ \tfrac{p}{2},\ 2p,\ \Lambda]$) **encadraient** ce coin sans le séparer.
Une règle de quadrature qui enjambe un point non lisse ne converge pas : elle
s'arrête sur l'erreur de placement de ses nœuds. La cellule suivante le mesure
en quatre cases, et l'erreur **change de valeur quand on monte la précision**
($2{,}08\times10^{-4}$ à 20 comme à 30 chiffres, $4{,}63\times10^{-5}$ à 45
comme à 60) **au lieu de tendre vers zéro** — signature d'un artefact de règle,
pas d'un terme physique.

> Les précisions et erreurs ci-dessus sont celles de `mpmath` à ses réglages de précision. En C#, le contrôle se fait à précision `double` fixe, en divisant le pas par deux. Un plateau dû au placement des nœuds n'exclut pas la convergence quand le pas tend vers zéro.

In [3]:
// Contrôle à quatre cases : variable angulaire et séparation du coin
// Variable d'origine ψ : k³/k² = k. On évalue le même dénominateur
// sous la forme (k−p)² + 4kp sin²(ψ/2), sans annulation par soustraction.
// À k = p, ψ = 0 reste singulier dans l'intégrande d'origine.
double PsiPondere(double k, double p, double psi)
{
    double d = k - p, s = Math.Sin(psi / 2.0);
    double den = d * d + 4.0 * k * p * s * s;
    if (den == 0.0) throw new DivideByZeroException("Intégrande en ψ au coin k = p, ψ = 0.");
    double sn = Math.Sin(psi);
    return 4.0 * Math.PI * k * sn * sn / den;
}

double[] RupturesRadiales(double lam, double p, bool rupture)
{
    if (!(p > 0.0 && lam >= 2.0 * p)) throw new ArgumentOutOfRangeException(nameof(lam));
    return rupture ? new[] { 0.0, 0.5 * p, p, 2.0 * p, lam }
                   : new[] { 0.0, 0.5 * p, 2.0 * p, lam };
}
double QuadRadiale(Func<double, double> radial, double[] bornes, int m)
{
    double somme = 0.0;
    for (int i = 1; i < bornes.Length; i++)
    {
        double a = bornes[i - 1], b = bornes[i];
        if (a == b) continue; // la dernière borne se répète pour Λ = 2p
        somme += Quad((g, d) => radial(g < d ? a + g : b - d), a, b, m);
    }
    return somme;
}
double BDirectPsi(double lam, double pCarre, bool rupture, int m)
{
    double p = Math.Sqrt(pCarre);
    var bornes = RupturesRadiales(lam, p, rupture);
    // La règle ouverte ne visite pas le bord angulaire : au point de rupture
    // en k, sonder explicitement l'intégrande original au bord ψ = 0.
    if (rupture) _ = PsiPondere(p, p, 0.0);
    return QuadRadiale(k => Quad((psi, reste) => PsiPondere(k, p, psi), 0.0, Math.PI, m), bornes, m);
}

// t = tan(ψ/2) : sin²(ψ)dψ = 8t² dt/(1+t²)³ et le dénominateur
// angulaire devient ((k−p)²+(k+p)²t²)/(1+t²).
// Après multiplication par k³, le préfacteur est 32πk. Au coin k=p,
// le facteur t² se simplifie avant évaluation sur [0,1].
// Sur [1,+∞), poser u=1/t et inclure |dt/du|=1/u² : il reste
// 32πk u² / ((1+u²)² ((k−p)²u²+(k+p)²)) sur [0,1].
double BDirect(double lam, double pCarre, bool rupture, int m)
{
    double p = Math.Sqrt(pCarre);
    var bornes = RupturesRadiales(lam, p, rupture);
    double RadialT(double k)
    {
        double d = k - p, s = k + p, d2 = d * d, s2 = s * s;
        double facteur = 32.0 * Math.PI * k;
        double bas = Quad((t, reste) =>
        {
            double t2 = t * t, a = 1.0 + t2;
            return d == 0.0 ? facteur / (a * a * s2) : facteur * t2 / (a * a * (d2 + s2 * t2));
        }, 0.0, 1.0, m);
        double queue = Quad((u, reste) =>
        {
            double u2 = u * u, a = 1.0 + u2;
            return facteur * u2 / (a * a * (d2 * u2 + s2));
        }, 0.0, 1.0, m);
        return bas + queue;
    }
    return QuadRadiale(RadialT, bornes, m);
}
BigInteger ExactDirectFixe(BigInteger lam, BigInteger pCarre) =>
    MultFixe(pi2Fixe, echelleFixe + LogFixe(lam * lam, pCarre));

// controle a quatre cases en Lam = 2 : deux variables x deux traitements du coin
const int Lam2 = 2;
// Python, cellule 6 : « valeur exacte », puis écarts absolus du tableau.
const string RefExact2 = "23.55178132880359170852945";
const string RefExact2Tol = "0.00000000000000000000001"; // une unité du dernier chiffre cité
const string RefNaive2 = "0.0002077"; // les deux lignes, sans rupture
const string RefStab2 = "0.0"; // ligne t = tan(psi/2), avec rupture
const string RefException2 = "ZeroDivisionError"; // ligne psi d'origine, avec rupture
var exact2Fixe = ExactDirectFixe(Lam2, p2);
double exact2 = DoubleFixe(exact2Fixe);
var ecartExact2 = BigInteger.Abs(exact2Fixe - LireFixe(RefExact2));
bool exact2Ok = ecartExact2 <= LireFixe(RefExact2Tol);
Console.WriteLine(Inv($"Valeur analytique π²(1 + ln Λ²) = {TexteFixe(exact2Fixe)} ; Python, cellule 6, cité = {RefExact2} ; écart = {DoubleFixe(ecartExact2):E6} : {Verdict(exact2Ok)}"));
Console.WriteLine(Inv($"Test avant calcul : |Q(h/2) − Q(h)| ≤ {TolQuad:G} ; même seuil pour l'accord à la valeur analytique. Les artefacts des voies naïves ne sont pas des cibles numériques."));
Verifier(exact2Ok, "La valeur analytique à Λ = 2 diffère de la citation.");

var controle2 = new List<(string Variable, bool Rupture, double Valeur, double Ecart, double Raffinement, decimal EcartCite)>();
bool exceptionCoin = false;
foreach (var voie in new (string Nom, Func<double, double, bool, int, double> Calcul)[]
{
    ("ψ d'origine", BDirectPsi), ("t = tan(ψ/2)", BDirect)
})
{
    foreach (bool rupture in new[] { false, true })
    {
        string cas = rupture ? "avec rupture k=p" : "sans rupture k=p";
        double qN, qFin;
        try
        {
            qN = voie.Calcul(Lam2, (double)p2, rupture, DemiMaillage);
            qFin = voie.Calcul(Lam2, (double)p2, rupture, DemiMaillageFin);
        }
        catch (DivideByZeroException ex) when (voie.Nom == "ψ d'origine" && rupture)
        {
            exceptionCoin = true;
            Console.WriteLine($"{voie.Nom}, {cas} : {ex.GetType().Name} interceptée ; Python, cellule 6, cité = {RefException2} ; même singularité.");
            continue;
        }
        double raff = Math.Abs(qFin - qN), ecart = Math.Abs(qFin - exact2);
        string cite = rupture ? RefStab2 : RefNaive2;
        decimal ecartCite = EcartCitation(ecart, cite);
        controle2.Add((voie.Nom, rupture, qFin, ecart, raff, ecartCite));
        Console.WriteLine(Inv($"{voie.Nom}, {cas} : {qN:G17} → {qFin:G17} ; raffinement = {raff:E6} ({Verdict(raff <= TolQuad)}) ; écart analytique = {ecart:E6} ({Verdict(ecart <= TolQuad)})"));
        Console.WriteLine(Inv($"  Écart analytique Python cité (cellule 6, même case) = {cite} ; différence d'amplitude C#–Python = {ecartCite:E6} ({Verdict(ecartCite <= (decimal)TolQuad)})"));
        if (rupture)
            Verifier(raff <= TolQuad && ecart <= TolQuad && ecartCite <= (decimal)TolQuad,
                "La voie t stabilisée ne satisfait pas le contrôle à Λ = 2.");
        else
            Verifier(double.IsFinite(qFin) && ecart > TolQuad, "Le témoin sans rupture ne met pas en évidence l'effet du coin à ce maillage.");
    }
}
Verifier(exceptionCoin && controle2.Count == 3, "Les quatre cases, dont l'exception au coin en ψ, sont requises.");
double accordNaif2 = Math.Abs(controle2[0].Valeur - controle2[1].Valeur);
Console.WriteLine(Inv($"Accord entre variables sans rupture : |Bψ − Bt| = {accordNaif2:E6} ; seuil = {TolQuad:G} : {Verdict(accordNaif2 <= TolQuad)}."));
Verifier(accordNaif2 <= TolQuad, "Les deux variables sans rupture devraient isoler le même artefact radial.");

Valeur analytique π²(1 + ln Λ²) = 23.55178132880359170852944930778434327791325750361661765706099 ; Python, cellule 6, cité = 23.55178132880359170852945 ; écart = 6.922157E-025 : CONFORME


Test avant calcul : |Q(h/2) − Q(h)| ≤ 1E-10 ; même seuil pour l'accord à la valeur analytique. Les artefacts des voies naïves ne sont pas des cibles numériques.


ψ d'origine, sans rupture k=p : 23.552658240853834 → 23.551989002291407 ; raffinement = 6.692386E-004 (ÉCART) ; écart analytique = 2.076735E-004 (ÉCART)


  Écart analytique Python cité (cellule 6, même case) = 0.0002077 ; différence d'amplitude C#–Python = 2.651219E-008 (ÉCART)


ψ d'origine, avec rupture k=p : DivideByZeroException interceptée ; Python, cellule 6, cité = ZeroDivisionError ; même singularité.


t = tan(ψ/2), sans rupture k=p : 23.552658240853837 → 23.551989002291407 ; raffinement = 6.692386E-004 (ÉCART) ; écart analytique = 2.076735E-004 (ÉCART)


  Écart analytique Python cité (cellule 6, même case) = 0.0002077 ; différence d'amplitude C#–Python = 2.651219E-008 (ÉCART)


t = tan(ψ/2), avec rupture k=p : 23.551781328803589 → 23.551781328803592 ; raffinement = 3.552714E-015 (CONFORME) ; écart analytique = 0.000000E+000 (CONFORME)


  Écart analytique Python cité (cellule 6, même case) = 0.0 ; différence d'amplitude C#–Python = 0.000000E+000 (CONFORME)


Accord entre variables sans rupture : |Bψ − Bt| = 0.000000E+000 ; seuil = 1E-10 : CONFORME.


### Le contrôle qui tranche : deux causes possibles, une seule vraie

*Référence Python (`mpmath`) :*

| variable | sans rupture en $k=p$ | avec rupture en $k=p$ |
|---|---|---|
| $\psi$ d'origine | $2{,}08\times10^{-4}$ d'écart | `ZeroDivisionError` |
| $t = \tan(\psi/2)$ | $2{,}08\times10^{-4}$ — **identique** | exact à $3\times10^{-29}$ |

Deux enseignements, tous deux mesurés et non supposés :

- **changer de variable ne change rien** tant que le coin n'est pas séparé :
  les deux routes rendent la même valeur, chiffre pour chiffre. L'erreur
  n'était donc pas dans l'intégrale angulaire, où on la cherchait ;
- **séparer le coin suffit, mais pas dans n'importe quelle variable** : en
  $\psi$, $k = p$ est une singularité de bord et la cellule lève
  `ZeroDivisionError` ; en $t$, la forme
  $\frac{32\pi\,t^2}{k^2(1+t^2)^2\left((k-p)^2+(k+p)^2t^2\right)}$ y est finie,
  parce que le $t^2$ du numérateur annule exactement le $t^2$ du dénominateur.

La voie directe ainsi stabilisée n'a **aucun résidu** : elle rend
$\pi^2(1+\ln\Lambda^2)$ à $3\times10^{-29}$ près, à toute coupure. Le résidu de
$1/\Lambda^2$ appartient à la **forme fermée** — au décalage de la section 2 —
et la cellule suivante sépare les deux par leur loi d'échelle, non par leur
valeur : l'artefact est **constant** en $\Lambda$, le résidu **décroît**.

> **Lecture C# :** sans rupture, l'accord des deux variables se teste au seuil affiché, pas chiffre pour chiffre avec `mpmath`. Avec rupture, la voie en $\psi$ lève et intercepte `DivideByZeroException`, tandis que la voie en $t$ teste le raffinement et l'accord à la valeur analytique à $10^{-10}$. Les écarts de quadrature du tableau Python ne sont pas des mesures du port ; le tableau suivant contrôle séparément leur comportement avec la coupure à maillage fixé.

In [4]:
// Échelle de l’artefact et comparaison des deux voies
// Lire les citations scientifiques sans perdre les écarts inférieurs à 10^-28.
BigInteger LireScientifiqueFixe(string texte)
{
    var champs = texte.Split(new[] { 'e', 'E' });
    if (champs.Length == 1) return LireFixe(texte);
    if (champs.Length != 2) throw new FormatException(texte);
    int exposant = int.Parse(champs[1], NumberStyles.Integer, culture);
    var mantisse = LireFixe(champs[0]);
    return exposant >= 0 ? mantisse * BigInteger.Pow(10, exposant)
                         : mantisse / BigInteger.Pow(10, -exposant);
}
BigInteger FixeDepuisDouble(double valeur) => LireScientifiqueFixe(valeur.ToString("G17", culture));
BigInteger EcartRefFixe(BigInteger valeur, string citation) =>
    BigInteger.Abs(valeur - LireScientifiqueFixe(citation));

// Python, cellule 8, premier tableau : écarts absolus stabilisé et naïf.
var referencesEchelle = new[] {
    (Lam: 2, Stab: "0.0", Naif: "0.000207673"),
    (Lam: 8, Stab: "6.31089e-30", Naif: "0.000207673"),
    (Lam: 64, Stab: "1.26218e-29", Naif: "0.000207673")
};
var seuilQuadFixe = LireScientifiqueFixe("1e-10");
var controleEchelle = new List<(int Lam, double Stable, double Naif, double ErrStable, double ErrNaif, double RaffStable, double RaffNaif, BigInteger EcartRefStable, BigInteger EcartRefNaif)>();
Console.WriteLine(Inv($"Écarts absolus à π²(1 + ln Λ²) ; tanh–sinh, {2 * DemiMaillage + 1} → {2 * DemiMaillageFin + 1} nœuds par intervalle ; seuil = {TolQuad:G}."));
foreach (var ligne in referencesEchelle)
{
    double exact = DoubleFixe(ExactDirectFixe(ligne.Lam, p2));
    double stableN = BDirect(ligne.Lam, (double)p2, true, DemiMaillage);
    double stable = BDirect(ligne.Lam, (double)p2, true, DemiMaillageFin);
    double naifN = BDirectPsi(ligne.Lam, (double)p2, false, DemiMaillage);
    double naif = BDirectPsi(ligne.Lam, (double)p2, false, DemiMaillageFin);
    double errStable = Math.Abs(stable - exact), errNaif = Math.Abs(naif - exact);
    double raffStable = Math.Abs(stable - stableN), raffNaif = Math.Abs(naif - naifN);
    var ecartRefStable = EcartRefFixe(FixeDepuisDouble(errStable), ligne.Stab);
    var ecartRefNaif = EcartRefFixe(FixeDepuisDouble(errNaif), ligne.Naif);
    controleEchelle.Add((ligne.Lam, stable, naif, errStable, errNaif, raffStable, raffNaif, ecartRefStable, ecartRefNaif));
    bool ok = errStable <= TolQuad && raffStable <= TolQuad && ecartRefStable <= seuilQuadFixe;
    Console.WriteLine(Inv($"Λ = {ligne.Lam,2} | stabilisée : {errStable:E6} ; raffinement = {raffStable:E6} ; Python cité = {ligne.Stab} ; |Δ citation| = {DoubleFixe(ecartRefStable):E6} : {Verdict(ok)}"));
    Console.WriteLine(Inv($"        | naïve      : {errNaif:E6} ; raffinement = {raffNaif:E6} ({Verdict(raffNaif <= TolQuad)}) ; Python cité = {ligne.Naif} ; |Δ citation| = {DoubleFixe(ecartRefNaif):E6} ({Verdict(ecartRefNaif <= seuilQuadFixe)})"));
    Verifier(ok && double.IsFinite(naif) && errNaif > TolQuad,
        Inv($"Contrôle de la stabilisation ou du témoin naïf en défaut à Λ = {ligne.Lam}."));
}
// À maillage fixé, le coin est dans le même sous-intervalle pour les trois Λ.
double dispersionNaive = controleEchelle.Max(l => l.ErrNaif) - controleEchelle.Min(l => l.ErrNaif);
Console.WriteLine(Inv($"Dispersion de l’erreur naïve entre coupures = {dispersionNaive:E6} ; seuil = {TolQuad:G} : {Verdict(dispersionNaive <= TolQuad)}."));
Verifier(dispersionNaive <= TolQuad, "Le témoin naïf ne présente pas le même écart entre les trois coupures à maillage fixé.");

// Second tableau de la cellule Python 8. Ses lignes numériques ne sont pas
// visibles dans l’extrait : citer la forme fermée de la cellule 4 (mêmes
// instances), et en Λ=2 la valeur analytique imprimée dans la cellule 6.
// Pour Λ=8, distinguer le contrôle analytique calculé d’une citation Python.
var comparaisonVoies = new List<(int Lam, BigInteger Fermee, double Directe, double Relatif, double Decalage, double EcartDirectAnalytique)>();
Console.WriteLine("Forme fermée contre voie directe : le terme de décalage");
foreach (var cas in casFermes)
{
    var ligne = controleEchelle.Single(l => l.Lam == cas.Lam);
    var fermee = BFerme(cas.Lam, p2);
    double fermeeD = DoubleFixe(fermee);
    double decalage = fermeeD - ligne.Stable, relatif = Math.Abs(decalage / fermeeD);
    double accordDirect = Math.Abs(ligne.Stable - DoubleFixe(ExactDirectFixe(cas.Lam, p2)));
    var ecartFermee = EcartRefFixe(fermee, cas.Ref);
    comparaisonVoies.Add((cas.Lam, fermee, ligne.Stable, relatif, decalage, accordDirect));
    Console.WriteLine(Inv($"Λ = {cas.Lam} | fermée = {TexteFixe(fermee)} | directe = {ligne.Stable:G17} | écart relatif = {relatif:G6}"));
    Console.WriteLine(Inv($"  Fermée : Python, cellule 4, cité = {cas.Ref} ; |Δ citation| = {DoubleFixe(ecartFermee):E6}. Directe : |Δ formule analytique| = {accordDirect:E6}."));
    if (cas.Lam == Lam2)
    {
        var ecartDirectCite = EcartRefFixe(FixeDepuisDouble(ligne.Stable), RefExact2);
        Console.WriteLine(Inv($"  Directe : valeur analytique Python, cellule 6, citée = {RefExact2} ; |Δ citation| = {DoubleFixe(ecartDirectCite):E6}."));
        Verifier(ecartDirectCite <= seuilQuadFixe, "La voie directe à Λ=2 ne rejoint pas la valeur analytique citée.");
    }
    Verifier(ecartFermee <= LireFixe(cas.Tol) && accordDirect <= TolQuad && decalage > TolQuad,
        "Le décalage doit être positif et distinct de l’erreur de quadrature.");
}
bool decroitDecalage = comparaisonVoies[1].Decalage < comparaisonVoies[0].Decalage
    && comparaisonVoies[1].Relatif < comparaisonVoies[0].Relatif;
Console.WriteLine(Inv($"Décalage absolu de Λ=2 à Λ=8 : {comparaisonVoies[0].Decalage:E6} → {comparaisonVoies[1].Decalage:E6} ; décroissance : {Verdict(decroitDecalage)}."));
Verifier(decroitDecalage, "Le décalage doit décroître entre les deux coupures.");

Écarts absolus à π²(1 + ln Λ²) ; tanh–sinh, 513 → 1025 nœuds par intervalle ; seuil = 1E-10.


Λ =  2 | stabilisée : 0.000000E+000 ; raffinement = 3.552714E-015 ; Python cité = 0.0 ; |Δ citation| = 0.000000E+000 : CONFORME


        | naïve      : 2.076735E-004 ; raffinement = 6.692386E-004 (ÉCART) ; Python cité = 0.000207673 ; |Δ citation| = 4.878145E-010 (ÉCART)


Λ =  8 | stabilisée : 0.000000E+000 ; raffinement = 7.105427E-015 ; Python cité = 6.31089e-30 ; |Δ citation| = 6.310890E-030 : CONFORME


        | naïve      : 2.076735E-004 ; raffinement = 6.692386E-004 (ÉCART) ; Python cité = 0.000207673 ; |Δ citation| = 4.878074E-010 (ÉCART)


Λ = 64 | stabilisée : 1.421085E-014 ; raffinement = 2.842171E-014 ; Python cité = 1.26218e-29 ; |Δ citation| = 1.421085E-014 : CONFORME


... (5 outputs left out)

  Directe : valeur analytique Python, cellule 6, citée = 23.55178132880359170852945 ; |Δ citation| = 2.914705E-016.


Λ = 8 | fermée = 50.967419252655640797151307565586231389532341197890495199244284 | directe = 50.916135184232061 | écart relatif = 0.00100621


  Fermée : Python, cellule 4, cité = 50.9674192526556407971513075656 ; |Δ citation| = 1.376861E-029. Directe : |Δ formule analytique| = 0.000000E+000.


Décalage absolu de Λ=2 à Λ=8 : 7.930207E-001 → 5.128407E-002 ; décroissance : CONFORME.


### Lecture — l'artefact est constant, le décalage décroît

Le premier tableau sépare deux comportements que leur seule valeur à $\Lambda = 64$ ne distingue pas. La voie directe **stabilisée** rejoint la valeur exacte à la précision de travail ; la voie **naïve** conserve le même écart absolu à toutes les coupures. Cet écart constant est donc un artefact de quadrature, pas un terme en $1/\Lambda^2$.

Le second tableau mesure, lui, un véritable écart entre la **forme fermée** et la **voie directe** : il passe d'environ $3{,}3\,\%$ à $\Lambda=2$ à $0{,}10\,\%$ à $\Lambda=8$. C'est le terme de décalage de la boule dure, dont la section 8 vérifiera la loi d'échelle — deux quantités différentes, séparées par leur comportement en $\Lambda$ et non par leur ressemblance ponctuelle.

> En C#, « le même écart » signifie ici que sa dispersion entre coupures reste sous la tolérance affichée, **à maillage fixé**. L'amplitude naïve dépend du placement des nœuds et n'est pas celle de `mpmath` ; le tableau affiche leur différence.

## 4. La divergence : $B(\Lambda) - \pi^2 \ln \Lambda^2$ tend vers une constante

Le comptage de puissances prédit $B(\Lambda) = \pi^2\big(\ln\Lambda^2 +
c\big) + O(1/\Lambda^2)$. Le tableau mesure la soustraction — la constante
émerge chiffre après chiffre :

In [5]:
// Partie finie aux quatre grandes coupures
// Pour p²=1, annuler symboliquement les logarithmes communs avant
// l’évaluation : B_ferme(Λ) − π² ln(Λ²) = π² + résidu(Λ).
BigInteger PartieFinieFixe(BigInteger lam) => pi2Fixe + Residufixe(lam, p2);

// Arrondir pour l’affichage uniquement ; les comparaisons gardent l’échelle 10^60.
string ArrondiTexteFixe(BigInteger valeur, int decimales)
{
    var unite = BigInteger.Pow(10, DecimalesFixes - decimales);
    var arrondi = (BigInteger.Abs(valeur) + unite / 2) / unite * unite;
    return TexteFixe(valeur.Sign < 0 ? -arrondi : arrondi);
}
// Python, cellule 11 : partie finie à 22 chiffres significatifs et ratio
// à 8 chiffres significatifs, ici inférieur à 1 et supérieur à 0,1.
var seuilPartieFinie = LireScientifiqueFixe("1e-21"); // une unité du dernier chiffre cité
var seuilRatioFinie = LireScientifiqueFixe("5e-9"); // demi-unité d’arrondi des ratios
var casFinie = new[] {
    (E: 2, Ref: "9.869933382968020057851", Ratio: "--"),
    (E: 4, Ref: "9.869604433988039906451", Ratio: "0.99996667"),
    (E: 6, Ref: "9.869604401092648486968", Ratio: "1.0"),
    (E: 8, Ref: "9.869604401089358947821", Ratio: "1.0")
};
var ecartsFinie = new List<(int E, BigInteger Valeur, BigInteger Ecart, BigInteger EcartRatio)>();
var finiePrecedente = BigInteger.Zero;
Console.WriteLine("     Lam          B(Lam) - pi^2 ln(Lam^2)   residu/precedent");
foreach (var cas in casFinie)
{
    var lam = BigInteger.Pow(10, cas.E);
    var finie = PartieFinieFixe(lam);
    var ecart = EcartRefFixe(finie, cas.Ref);
    bool precedentExiste = !finiePrecedente.IsZero;
    var ratio = precedentExiste ? DivFixe(finie, finiePrecedente) : BigInteger.Zero;
    var ecartRatio = precedentExiste ? EcartRefFixe(ratio, cas.Ratio) : BigInteger.Zero;
    string ratioTexte = precedentExiste ? ArrondiTexteFixe(ratio, 8) : "--";
    if (ratioTexte == "1") ratioTexte = "1.0";
    bool okValeur = ecart <= seuilPartieFinie && finie > pi2Fixe;
    bool okRatio = !precedentExiste || (ratio < echelleFixe && ecartRatio <= seuilRatioFinie);
    ecartsFinie.Add((cas.E, finie, ecart, ecartRatio));
    Console.WriteLine(Inv($"Lam=1e{cas.E}  B - pi^2 ln(Lam^2) = {ArrondiTexteFixe(finie, 21)}  ratio = {ratioTexte}"));
    Console.WriteLine(Inv($"  Python, cellule 11, cité = {cas.Ref} ; |Δ valeur| = {DoubleFixe(ecart):E6} ; seuil = {DoubleFixe(seuilPartieFinie):E1} : {Verdict(okValeur)}"));
    if (precedentExiste)
        Console.WriteLine(Inv($"  Ratio Python cité = {cas.Ratio} ; |Δ ratio| = {DoubleFixe(ecartRatio):E6} ; seuil = {DoubleFixe(seuilRatioFinie):E1} : {Verdict(okRatio)}"));
    Verifier(okValeur && okRatio, "La partie finie ou son rapport ne satisfait pas le contrôle.");
    finiePrecedente = finie;
}

     Lam          B(Lam) - pi^2 ln(Lam^2)   residu/precedent


Lam=1e2  B - pi^2 ln(Lam^2) = 9.869933382968020057851  ratio = --


  Python, cellule 11, cité = 9.869933382968020057851 ; |Δ valeur| = 2.853838E-022 ; seuil = 1.0E-021 : CONFORME


Lam=1e4  B - pi^2 ln(Lam^2) = 9.869604433988039906451  ratio = 0.99996667


  Python, cellule 11, cité = 9.869604433988039906451 ; |Δ valeur| = 2.182126E-024 ; seuil = 1.0E-021 : CONFORME


  Ratio Python cité = 0.99996667 ; |Δ ratio| = 1.610920E-009 ; seuil = 5.0E-009 : CONFORME


... (2 outputs left out)

  Ratio Python cité = 1.0 ; |Δ ratio| = 3.333000E-009 ; seuil = 5.0E-009 : CONFORME


Lam=1e8  B - pi^2 ln(Lam^2) = 9.869604401089358947821  ratio = 1.0


  Python, cellule 11, cité = 9.869604401089358947821 ; |Δ valeur| = 3.043695E-022 ; seuil = 1.0E-021 : CONFORME


  Ratio Python cité = 1.0 ; |Δ ratio| = 3.333000E-013 ; seuil = 5.0E-009 : CONFORME


## 5. Planted truth n° 1 — le coefficient de $\ln \Lambda^2$ est $\pi^2$

La différence $B(10\Lambda) - B(\Lambda)$ isole le coefficient sans aucune
connaissance de la constante : divisée par $\ln 100$, elle doit rendre $\pi^2$
exactement.

> Pour la forme fermée à coupure finie, le quotient conserve la différence des deux résidus divisée par $\ln 100$. Le calcul à échelle fixe mesure cette correction au lieu de l'arrondir à zéro.

In [6]:
// Mesure du coefficient logarithmique à L = 10⁶
// Python, cellule 13 : coefficient, π² et écart relatif imprimés.
const string RefCoef13 = "9.86960440108865137690404751611";
const string RefPi13 = "9.86960440108935861883449099988";
const string RefRel13 = "7.16585895140256955602111645009e-14";
// La source soustrait deux valeurs d’ordre 10² à 30 chiffres ; le bruit
// se propage dans le quotient et dans son écart à π².
var seuilCoef13 = LireScientifiqueFixe("1e-28");
var seuilPi13 = LireScientifiqueFixe("1e-29");
var seuilRel13 = LireScientifiqueFixe("2e-29");
var seuilAsymptotique13 = LireScientifiqueFixe("1e-12"); // à L=10⁶, correction d’ordre L^-2
var lamCoef = BigInteger.Pow(10, 6);
var coefFixe = DivFixe(BFerme(10 * lamCoef, p2) - BFerme(lamCoef, p2), LogFixe(100, 1));
var coefRelFixe = DivFixe(BigInteger.Abs(coefFixe - pi2Fixe), pi2Fixe);
var ecartCoef13 = EcartRefFixe(coefFixe, RefCoef13);
var ecartPi13 = EcartRefFixe(pi2Fixe, RefPi13);
var ecartRel13 = EcartRefFixe(coefRelFixe, RefRel13);
void AfficherComparaisonFixe(string libelle, BigInteger valeur, string citation, BigInteger ecart, BigInteger seuil)
{
    Console.WriteLine(Inv($"{libelle} = {TexteFixe(valeur)} ; Python cité = {citation} ; |Δ| = {DoubleFixe(ecart):E6} ; seuil = {DoubleFixe(seuil):E2} : {Verdict(ecart <= seuil)}"));
}
AfficherComparaisonFixe("coefficient mesuré (cellule Python 13)", coefFixe, RefCoef13, ecartCoef13, seuilCoef13);
AfficherComparaisonFixe("π² (cellule Python 13)", pi2Fixe, RefPi13, ecartPi13, seuilPi13);
AfficherComparaisonFixe("écart relatif (cellule Python 13)", coefRelFixe, RefRel13, ecartRel13, seuilRel13);
bool coefAsymptotiqueOk = coefFixe < pi2Fixe && coefRelFixe <= seuilAsymptotique13;
Console.WriteLine(Inv($"Approche de π² par valeurs inférieures ; seuil relatif = {DoubleFixe(seuilAsymptotique13):E1} : {Verdict(coefAsymptotiqueOk)}."));
Verifier(ecartCoef13 <= seuilCoef13 && ecartPi13 <= seuilPi13 && ecartRel13 <= seuilRel13 && coefAsymptotiqueOk,
    "Le contrôle du coefficient logarithmique a échoué.");

coefficient mesuré (cellule Python 13) = 9.86960440108865137690404751611014151099877434098144703442847 ; Python cité = 9.86960440108865137690404751611 ; |Δ| = 1.415110E-031 ; seuil = 1.00E-028 : CONFORME


π² (cellule Python 13) = 9.86960440108935861883449099987615113531369940724079062641332 ; Python cité = 9.86960440108935861883449099988 ; |Δ| = 3.848865E-030 ; seuil = 1.00E-029 : CONFORME


écart relatif (cellule Python 13) = 0.000000000000071658589514025695286124897151745022858088918518 ; Python cité = 7.16585895140256955602111645009e-14 ; |Δ| = 2.740863E-031 ; seuil = 2.00E-029 : CONFORME


Approche de π² par valeurs inférieures ; seuil relatif = 1.0E-012 : CONFORME.


## 6. Planted truth n° 2 — la constante finie est $\pi^2$, ancrée dans l'intégrale $x$

Le développement de la forme fermée donne
$B(\Lambda) = \pi^2\big(\ln\Lambda^2 + \int_0^1 \ln\frac{1}{x(1-x)}\,dx
- 1\big) + o(1)$. L'intégrale $x$ se calcule **à la main** :
$\int_0^1 \ln x\,dx = \int_0^1 \ln(1-x)\,dx = -1$, donc la constante vaut
$\pi^2(2 - 1) = \pi^2$. Les deux cellules mesurent l'ancre et la limite par
deux routes indépendantes : la forme fermée à grande $\Lambda$, et la voie
directe tronquée.

La distinction que ce carnet confondait, et que la section 3 a démêlée : la
**voie directe n'a aucun résidu** — elle converge vers $\pi^2$ ; la **forme
fermée en porte un**, $\pi^2 p^2/(3\Lambda^2)$, qui vient du décalage de la
section 2 et reste dans la voie fermée quelle que soit la précision de la
quadrature. Ce sont deux quantités de natures différentes, et la section 8 les
sépare par leur loi d'échelle.

In [7]:
// ancre : l'integrale x, calculee par quadrature (et verifiee a la main : -1 + -1 = -2)
// Python, cellule 15, première ligne : « int_0^1 ln(1/(x(1-x))) dx ».
const string RefAncre15 = "2.0";
var ancreExacte = new BigInteger(2);
// Le schéma tanh–sinh fournit x et 1−x sans soustraire deux doubles
// presque égaux ; la fonction évaluée est l’intégrande d’origine.
double IntegrandeAncre(double x, double unMoinsX) => Math.Log(1.0 / (x * unMoinsX));
Console.WriteLine(Inv($"Ancre : tanh–sinh sur [0,1], h = {BorneTanhSinh / DemiMaillage:G} → {BorneTanhSinh / DemiMaillageFin:G}, {2 * DemiMaillage + 1} → {2 * DemiMaillageFin + 1} nœuds ; seuil absolu = {TolQuad:G}."));
double ancreN = Quad(IntegrandeAncre, 0.0, 1.0, DemiMaillage);
double intX = Quad(IntegrandeAncre, 0.0, 1.0, DemiMaillageFin);
double raffAncre = Math.Abs(intX - ancreN), ecartAncreAnalytique = Math.Abs(intX - (double)ancreExacte);
var ecartAncreCite = EcartRefFixe(FixeDepuisDouble(intX), RefAncre15);
bool ancreOk = raffAncre <= TolQuad && ecartAncreAnalytique <= TolQuad && ecartAncreCite <= seuilQuadFixe;
Console.WriteLine(Inv($"int_0^1 ln(1/(x(1-x))) dx = {intX:G17} ; à la main = {ancreExacte} ; Python cité = {RefAncre15} ; |Δ citation| = {DoubleFixe(ecartAncreCite):E6}"));
Console.WriteLine(Inv($"  Raffinement : {ancreN:G17} → {intX:G17} ; |Δ raffinement| = {raffAncre:E6} ; |Δ analytique| = {ecartAncreAnalytique:E6} : {Verdict(ancreOk)}"));
Verifier(ancreOk, "La quadrature de l’intégrale d’ancrage a échoué.");

// limite grande Lambda
// Python, cellule 15 : trois dernières lignes imprimées. La soustraction
// Python à 30 chiffres porte sur des quantités d’ordre 10² : sa petite
// différence relative ne possède pas 30 chiffres de précision absolue.
const string RefFNum15 = "9.86960440108935861883449428972";
const string RefFTheo15 = "9.86960440108935861883449099988";
const string RefFRel15 = "3.33330510716480601405036945999e-25";
var seuilFNum15 = LireScientifiqueFixe("1e-28");
var seuilFTheo15 = LireScientifiqueFixe("1e-29");
var seuilFRel15 = LireScientifiqueFixe("2e-29");
// F déduit de la quadrature est un contrôle indépendant en double.
// Pour résoudre le résidu à Λ=10¹², employer l’ancre analytique exacte :
// injecter ici l’erreur de quadrature masquerait entièrement le résidu.
double fTheoQuad = Math.PI * Math.PI * (intX - 1.0);
var fTheo = pi2Fixe * (ancreExacte - BigInteger.One);
var fNum = PartieFinieFixe(BigInteger.Pow(10, 12));
var fRel = DivFixe(BigInteger.Abs(fNum - fTheo), fTheo);
var ecartFNum15 = EcartRefFixe(fNum, RefFNum15);
var ecartFTheo15 = EcartRefFixe(fTheo, RefFTheo15);
var ecartFRel15 = EcartRefFixe(fRel, RefFRel15);
var ecartFTheoQuad15 = EcartRefFixe(FixeDepuisDouble(fTheoQuad), RefFTheo15);
// Propagation du seuil sur l’ancre par le facteur π², plus le seuil
// absolu de comparaison des évaluations en double.
double seuilFTheoQuad15 = TolQuad * (1.0 + DoubleFixe(pi2Fixe));
bool fTheoQuadOk = DoubleFixe(ecartFTheoQuad15) <= seuilFTheoQuad15;
Console.WriteLine(Inv($"F déduit de l’ancre quadraturée = {fTheoQuad:G17} ; Python cité = {RefFTheo15} ; |Δ citation| = {DoubleFixe(ecartFTheoQuad15):E6} ; seuil propagé = {seuilFTheoQuad15:E6} : {Verdict(fTheoQuadOk)}"));
AfficherComparaisonFixe("F mesuré (Λ=1e12, cellule Python 15)", fNum, RefFNum15, ecartFNum15, seuilFNum15);
AfficherComparaisonFixe("F théorique, ancre exacte (cellule Python 15)", fTheo, RefFTheo15, ecartFTheo15, seuilFTheo15);
AfficherComparaisonFixe("écart relatif à échelle fixe (cellule Python 15)", fRel, RefFRel15, ecartFRel15, seuilFRel15);
Verifier(fTheoQuadOk && fNum > fTheo && ecartFNum15 <= seuilFNum15 && ecartFTheo15 <= seuilFTheo15 && ecartFRel15 <= seuilFRel15,
    "Le contrôle de la constante finie et de son résidu a échoué.");

Ancre : tanh–sinh sur [0,1], h = 0.015625 → 0.0078125, 513 → 1025 nœuds ; seuil absolu = 1E-10.


int_0^1 ln(1/(x(1-x))) dx = 2 ; à la main = 2 ; Python cité = 2.0 ; |Δ citation| = 0.000000E+000


  Raffinement : 2 → 2 ; |Δ raffinement| = 0.000000E+000 ; |Δ analytique| = 0.000000E+000 : CONFORME


F déduit de l’ancre quadraturée = 9.869604401089358 ; Python cité = 9.86960440108935861883449099988 ; |Δ citation| = 6.188345E-016 ; seuil propagé = 1.086960E-009 : CONFORME


F mesuré (Λ=1e12, cellule Python 15) = 9.86960440108935861883449428974428483176657235207063043824364 ; Python cité = 9.86960440108935861883449428972 ; |Δ| = 2.428483E-029 ; seuil = 1.00E-028 : CONFORME


F théorique, ancre exacte (cellule Python 15) = 9.86960440108935861883449099987615113531369940724079062641332 ; Python cité = 9.86960440108935861883449099988 ; |Δ| = 3.848865E-030 ; seuil = 1.00E-029 : CONFORME


écart relatif à échelle fixe (cellule Python 15) = 0.000000000000000000000000333333333333333333333333283333333332 ; Python cité = 3.33330510716480601405036945999e-25 ; |Δ| = 2.822617E-030 ; seuil = 2.00E-029 : CONFORME


In [8]:
// F par la voie directe : version stabilisee, puis le temoin naif
const int L64 = 64;
// Python, cellule 16 : colonnes « F mesure (Lambda=64) » et « Ecart relatif a F_theo ».
var refsF16 = new[] {
    (Voie: "Directe stabilisée", Stable: true, F: "9.86960440108935861883449099989", Rel: "1.27885e-30"),
    (Voie: "Naïve, coin k=p non séparé", Stable: false, F: "9.86981207457717563409161549917", Rel: "2.10417e-5")
};
// On reprend les quadratures aux deux maillages de la cellule 8 :
// soustraire le même logarithme aux deux résultats ne change pas leur écart.
var directe64 = controleEchelle.Single(ligne => ligne.Lam == L64);
var soustraction64Fixe = MultFixe(pi2Fixe, LogFixe((BigInteger)L64 * L64, BigInteger.One));
double soustraction64 = DoubleFixe(soustraction64Fixe);
// Même ancre quadraturée que dans la comparaison Python ; le seuil relatif
// est le seuil absolu ramené à la taille de l'ancre F.
double seuilRelF16 = TolQuad / Math.Abs(fTheoQuad);
var seuilRelF16Fixe = FixeDepuisDouble(seuilRelF16);
var ecartsF16 = new List<(bool Stable, double F, double Relatif, double Raffinement, BigInteger EcartF, BigInteger EcartRelatif)>();
Console.WriteLine(Inv($"Λ = {L64} ; F théorique issu de l'ancre quadraturée = {fTheoQuad:G17} ; seuil absolu = {TolQuad:G} ; seuil relatif = {seuilRelF16:E6}."));
foreach (var reference in refsF16)
{
    double b = reference.Stable ? directe64.Stable : directe64.Naif;
    double raff = reference.Stable ? directe64.RaffStable : directe64.RaffNaif;
    double f = b - soustraction64;
    double rel = Math.Abs((f - fTheoQuad) / fTheoQuad);
    var ecartF = EcartRefFixe(FixeDepuisDouble(f), reference.F);
    var ecartRel = EcartRefFixe(FixeDepuisDouble(rel), reference.Rel);
    bool accordF = ecartF <= seuilQuadFixe, accordRel = ecartRel <= seuilRelF16Fixe;
    ecartsF16.Add((reference.Stable, f, rel, raff, ecartF, ecartRel));
    Console.WriteLine(Inv($"{reference.Voie} | F mesuré = {f:G17} | écart relatif à F théorique = {rel:G6}"));
    Console.WriteLine(Inv($"  Raffinement |ΔF| = {raff:E6} : {Verdict(raff <= TolQuad)} ; accord à l'ancre : {Verdict(rel <= seuilRelF16)}."));
    Console.WriteLine(Inv($"  F Python cité (cellule 16) = {reference.F} ; |Δ citation| = {DoubleFixe(ecartF):E6} : {Verdict(accordF)}."));
    Console.WriteLine(Inv($"  Écart relatif Python cité (cellule 16) = {reference.Rel} ; |Δ citation| = {DoubleFixe(ecartRel):E6} : {Verdict(accordRel)}."));
    if (reference.Stable)
        Verifier(accordF && accordRel && raff <= TolQuad && rel <= seuilRelF16,
            "La partie finie directe stabilisée ne rejoint pas l'ancre ou la référence.");
    else
    {
        Console.WriteLine("  Témoin du coin non séparé : l'amplitude dépend des nœuds ; un seuil manqué reste affiché comme un écart.");
        Verifier(double.IsFinite(rel) && rel > seuilRelF16,
            "Le témoin naïf doit laisser un écart à l'ancre détectable au maillage choisi.");
    }
}

Λ = 64 ; F théorique issu de l'ancre quadraturée = 9.869604401089358 ; seuil absolu = 1E-10 ; seuil relatif = 1.013212E-011.


Directe stabilisée | F mesuré = 9.8696044010893615 | écart relatif à F théorique = 3.59965E-16


  Raffinement |ΔF| = 2.842171E-014 : CONFORME ; accord à l'ancre : CONFORME.


  F Python cité (cellule 16) = 9.86960440108935861883449099989 ; |Δ citation| = 2.881166E-015 : CONFORME.


  Écart relatif Python cité (cellule 16) = 1.27885e-30 ; |Δ citation| = 3.599652E-016 : CONFORME.


Naïve, coin k=p non séparé | F mesuré = 9.8698120745771689 | écart relatif à F théorique = 2.10417E-05


  Raffinement |ΔF| = 6.692386E-004 : ÉCART ; accord à l'ancre : ÉCART.


  F Python cité (cellule 16) = 9.86981207457717563409161549917 ; |Δ citation| = 6.734092E-015 : CONFORME.


  Écart relatif Python cité (cellule 16) = 2.10417e-5 ; |Δ citation| = 2.359614E-011 : ÉCART.


  Témoin du coin non séparé : l'amplitude dépend des nœuds ; un seuil manqué reste affiché comme un écart.


### Lecture — les deux écarts ne sont pas interchangeables

*Lecture du tableau Python (`mpmath`) :*

La voie directe stabilisée reproduit $F_{\mathrm{théo}}=\pi^2$ à la précision de travail. Le témoin naïf porte au contraire l'artefact absolu de la section 3, constant en $\Lambda$ : ramené à $F=\pi^2$, il apparaît ici comme un écart **relatif** d'environ $2{,}1\times10^{-5}$.

Ce nombre n'est donc **pas** le résidu de la forme fermée. La confusion était possible parce que, à $\Lambda=64$, artefact et résidu ont des ordres de grandeur voisins ; les tableaux précédents les séparent par leur loi d'échelle, sans modifier aucune des mesures.

> **Lecture C# :** le tableau précédent mesure l'écart relatif à l'ancre pour chacune des deux voies. La stabilisation est contrôlée au seuil affiché ; l'amplitude du témoin naïf dépend du maillage `double` et n'est pas celle citée pour `mpmath`. Le tableau affiche cette différence sans l'attribuer au décalage de la boule.

## 7. La régularisation soustractive : un paramètre, une échelle, une limite finie

Le geste du carnet 01 face à ses limites, transposé : on **soustrait** le
divergent à une échelle $\mu$,

$$B_R(\mu^2) = \lim_{\Lambda \to \infty}\; B(\Lambda) - \pi^2 \ln\frac{\Lambda^2}{\mu^2} = \pi^2\big(1 + \ln\mu^2\big) \quad (p^2 = 1).$$

La limite existe, est finie, et **dépend de l'échelle** : c'est la première
apparition, dans la série, d'une quantité qui « court » —
$\partial B_R / \partial \ln \mu^2 = \pi^2$, mesuré dans la sortie.

In [9]:
// Soustraction à l’échelle μ et mesure de la pente
// Dans B_ferme − π² ln(Λ²/μ²), combiner d'abord les logarithmes :
// ln(Λ²/p²) − ln(Λ²/μ²) = ln(μ²/p²). La correction de coupure reste présente.
BigInteger BRFixe(BigInteger mu2, BigInteger lam, BigInteger pCarre) =>
    MultFixe(pi2Fixe, echelleFixe + LogFixe(mu2, pCarre) + CorrectionCrochetFixe(lam, pCarre));

// Python, cellule 19 : les deux colonnes des lignes « mu^2 » et la pente finale.
var casBR19 = new[] {
    (Mu2: 1, BR: "9.86960440108935861883449428972", Limite: "9.86960440108935861883449099988"),
    (Mu2: 4, BR: "23.5517813288035917085294525976", Limite: "23.5517813288035917085294493078"),
    (Mu2: 16, BR: "37.2339582565178247982244109056", Limite: "37.2339582565178247982244076157")
};
const string RefPente19 = "9.86960440108935861883449099986";
const string RefPi19 = "9.86960440108935861883449099988";
// Une unité du dernier chiffre des valeurs Python d'ordre 10² avant
// soustraction ; le seuil interne tient compte des troncatures à 10^-60.
var seuilBR19 = LireScientifiqueFixe("1e-28");
var seuilInterne19 = LireScientifiqueFixe("1e-55");
var lamBR19 = BigInteger.Pow(10, 12);
var residuBR19 = Residufixe(lamBR19, p2);
var ecartsBR19 = new List<(int Mu2, BigInteger Valeur, BigInteger EcartBR, BigInteger EcartLimite, BigInteger EcartAuResidu)>();
foreach (var cas in casBR19)
{
    var valeur = BRFixe(cas.Mu2, lamBR19, p2);
    var limite = MultFixe(pi2Fixe, echelleFixe + LogFixe(cas.Mu2, 1));
    var ecartBR = EcartRefFixe(valeur, cas.BR);
    var ecartLimite = EcartRefFixe(limite, cas.Limite);
    var correction = valeur - limite;
    var ecartAuResidu = BigInteger.Abs(correction - residuBR19);
    bool okCorrection = correction.Sign > 0 && ecartAuResidu <= seuilInterne19;
    ecartsBR19.Add((cas.Mu2, valeur, ecartBR, ecartLimite, ecartAuResidu));
    Console.WriteLine(Inv($"mu^2 = {cas.Mu2,2}"));
    AfficherComparaisonFixe("  B_R (cellule Python 19)", valeur, cas.BR, ecartBR, seuilBR19);
    AfficherComparaisonFixe("  π²(1 + ln μ²) (cellule Python 19)", limite, cas.Limite, ecartLimite, seuilBR19);
    Console.WriteLine(Inv($"  B_R − limite = {DoubleFixe(correction):E6} ; résidu calculé = {DoubleFixe(residuBR19):E6} ; |Δ au résidu| = {DoubleFixe(ecartAuResidu):E6} ; seuil interne = {DoubleFixe(seuilInterne19):E1} : {Verdict(okCorrection)}."));
    Verifier(ecartBR <= seuilBR19 && ecartLimite <= seuilBR19 && okCorrection,
        Inv($"Contrôle de la soustraction en défaut pour μ² = {cas.Mu2}."));
}

// dependance d'echelle mesuree par differences finies
var penteBR19 = DivFixe(ecartsBR19[1].Valeur - ecartsBR19[0].Valeur, LogFixe(4, 1));
var ecartPente19 = EcartRefFixe(penteBR19, RefPente19);
var ecartPi19 = EcartRefFixe(pi2Fixe, RefPi19);
var ecartPenteTheo19 = BigInteger.Abs(penteBR19 - pi2Fixe);
AfficherComparaisonFixe("dB_R/d ln(μ²) mesuré (cellule Python 19)", penteBR19, RefPente19, ecartPente19, seuilBR19);
AfficherComparaisonFixe("π² (cellule Python 19)", pi2Fixe, RefPi19, ecartPi19, seuilBR19);
Console.WriteLine(Inv($"|Pente − π²| = {DoubleFixe(ecartPenteTheo19):E6} ; seuil interne = {DoubleFixe(seuilInterne19):E1} : {Verdict(ecartPenteTheo19 <= seuilInterne19)}."));
Verifier(ecartPente19 <= seuilBR19 && ecartPi19 <= seuilBR19 && ecartPenteTheo19 <= seuilInterne19,
    "Le contrôle de la pente de B_R a échoué.");

mu^2 =  1


  B_R (cellule Python 19) = 9.86960440108935861883449428974428483176657235207063043824364 ; Python cité = 9.86960440108935861883449428972 ; |Δ| = 2.428483E-029 ; seuil = 1.00E-028 : CONFORME


  π²(1 + ln μ²) (cellule Python 19) = 9.86960440108935861883449099987615113531369940724079062641332 ; Python cité = 9.86960440108935861883449099988 ; |Δ| = 3.848865E-030 ; seuil = 1.00E-028 : CONFORME


  B_R − limite = 3.289868E-024 ; résidu calculé = 3.289868E-024 ; |Δ au résidu| = 0.000000E+000 ; seuil interne = 1.0E-055 : CONFORME.


mu^2 =  4


  B_R (cellule Python 19) = 23.55178132880359170852945259765247697436613044844645746889131 ; Python cité = 23.5517813288035917085294525976 ; |Δ| = 5.247697E-029 ; seuil = 1.00E-028 : CONFORME


... (5 outputs left out)

  B_R − limite = 3.289868E-024 ; résidu calculé = 3.289868E-024 ; |Δ au résidu| = 0.000000E+000 ; seuil interne = 1.0E-055 : CONFORME.


dB_R/d ln(μ²) mesuré (cellule Python 19) = 9.869604401089358618834490999876151135313699407240790626413319 ; Python cité = 9.86960440108935861883449099986 ; |Δ| = 1.615114E-029 ; seuil = 1.00E-028 : CONFORME


π² (cellule Python 19) = 9.86960440108935861883449099987615113531369940724079062641332 ; Python cité = 9.86960440108935861883449099988 ; |Δ| = 3.848865E-030 ; seuil = 1.00E-028 : CONFORME


|Pente − π²| = 1.000000E-060 ; seuil interne = 1.0E-055 : CONFORME.


## 8. Refus documentés (reconnaissance de constantes, anneau déclaré avant recherche)

Avant toute recherche de relation, l'anneau des constantes est déclaré :
$\{\pi, \ln 2, \gamma_E, \ln 4\pi\}$.

- **$B(\Lambda)/\ln\Lambda^2 \to \pi^2$** — et non $1$ : le coefficient de
  $\ln\Lambda^2$ mesuré par la planted truth n° 1 **est** $\pi^2$, donc
  $B \sim \pi^2 \ln\Lambda^2$ et le rapport tend vers $\pi^2$. Chercher une
  autre constante multiplicative serait fabriquer un résultat.
- **La partie finie $F = \pi^2$ ne contient ni $\gamma_E$ ni $\ln 4\pi$** —
  ces constantes appartiennent au schéma de régularisation dimensionnelle,
  citée ('t Hooft–Veltman 1979, déjà aux Sources de la série) et non reproduite
  ici : la boule dure les ignore, et c'est une **différence de schéma**, pas
  une contradiction.
- **Le résidu de la *forme fermée* $R(\Lambda) = B_{\mathrm{fermé}}(\Lambda) -
  \pi^2(\ln\Lambda^2+1)$ décroît en $1/\Lambda^2$, de coefficient $\pi^2/3$** —
  c'est le terme de décalage de la section 2 : deux contributions
  $\bar\Delta/\Lambda^2 = p^2/(6\Lambda^2)$, une par propagateur. Vérifié au
  ratio ci-dessous plutôt qu'affirmé.

*Lecture des mesures Python (`mpmath`) :*

- **La voie directe, elle, n'a aucun résidu** : stabilisée, elle rend
  $\pi^2(1+\ln\Lambda^2)$ à $3\times10^{-29}$ près, à toute coupure. Le
  $2\times10^{-5}$ que la première version de ce carnet publiait en section 6
  n'était pas ce résidu — c'était l'artefact de quadrature de la section 3. Les
  deux quantités sont du même ordre à $\Lambda = 64$ ($2{,}1\times10^{-5}$
  relatif pour l'artefact, $8{,}1\times10^{-5}$ pour le résidu), ce qui est
  exactement pourquoi la confusion pouvait passer inaperçue ; **leur loi
  d'échelle les sépare** : l'artefact ne bouge pas avec $\Lambda$, le résidu
  décroît en $1/\Lambda^2$.

> **Contrôles C# :** les seuils des quadratures sont ceux des tableaux précédents ; les valeurs à $10^{-29}$ appartiennent à `mpmath`. Le rapport $R/\mathrm{prédiction}$ teste le coefficient, tandis que le quotient des résidus à deux coupures teste l'exposant indépendamment du coefficient.

In [10]:
// Résidu de décalage : contrôler le coefficient et l’exposant
// Python, cellule 21 : tableau R, prédiction et rapport ; puis lignes finales.
// Les seuils par ligne couvrent une unité du dernier chiffre des mantisses.
var casResidu21 = new[] {
    (E: 2, R: "0.000328981878661439", Pred: "0.0003289868133696453", Rapport: "0.999985000286", TolR: "1e-18", TolPred: "1e-19"),
    (E: 3, R: "3.289867640216327e-6", Pred: "3.289868133696453e-6", Rapport: "0.99999985", TolR: "1e-21", TolPred: "1e-21"),
    (E: 4, R: "3.289868128761651e-8", Pred: "3.289868133696453e-8", Rapport: "0.9999999985", TolR: "1e-23", TolPred: "1e-23")
};
const string RefRatio21 = "9999.85001785685833284495861";
const string RefRel21 = "0.0000149982143141667155041389997736";
const string RefSeuil21 = "0.0005";
const string RefVerdict21 = "CONFORME";
var seuilRapportCite21 = LireScientifiqueFixe("1e-12");
// La soustraction Python d'ordres 10² à 30 chiffres peut perturber R(1e4)
// d'environ 10^-28 ; la division par R(1e4) amplifie cette erreur dans le ratio.
var seuilRatioCite21 = LireScientifiqueFixe("1e-16");
var seuilRelCite21 = LireScientifiqueFixe("1e-20");
var seuilCoefficient21 = LireScientifiqueFixe("2e-5"); // correction d'environ 1,5e-5 au plus petit Λ
var seuilExposant21 = LireScientifiqueFixe("5e-4");
var ecartsResidu21 = new List<(int E, BigInteger R, BigInteger EcartR, BigInteger EcartPred, BigInteger EcartRapport)>();
Console.WriteLine("   Lam                 R(Lam)     pi^2 p^2/(3 Lam^2)      rapport");
foreach (var cas in casResidu21)
{
    var lam = BigInteger.Pow(10, cas.E);
    var r = Residufixe(lam, p2);
    var pred = pi2Fixe * p2 / (3 * lam * lam);
    var rapport = DivFixe(r, pred);
    var ecartR = EcartRefFixe(r, cas.R);
    var ecartPred = EcartRefFixe(pred, cas.Pred);
    var ecartRapport = EcartRefFixe(rapport, cas.Rapport);
    var tolR = LireScientifiqueFixe(cas.TolR);
    var tolPred = LireScientifiqueFixe(cas.TolPred);
    var ecartCoefficient = BigInteger.Abs(rapport - echelleFixe);
    bool okCoefficient = r.Sign > 0 && ecartCoefficient <= seuilCoefficient21;
    ecartsResidu21.Add((cas.E, r, ecartR, ecartPred, ecartRapport));
    Console.WriteLine(Inv($"   1e{cas.E} {DoubleFixe(r),22:G16} {DoubleFixe(pred),22:G16} {ArrondiTexteFixe(rapport, 12),12}"));
    Console.WriteLine(Inv($"  Python, cellule 21, cités dans l’ordre : {cas.R}, {cas.Pred}, {cas.Rapport}."));
    Console.WriteLine(Inv($"  |ΔR| = {DoubleFixe(ecartR):E6} (seuil {cas.TolR}) : {Verdict(ecartR <= tolR)} ; |Δ prédiction| = {DoubleFixe(ecartPred):E6} (seuil {cas.TolPred}) : {Verdict(ecartPred <= tolPred)} ; |Δ rapport| = {DoubleFixe(ecartRapport):E6} (seuil {DoubleFixe(seuilRapportCite21):E1}) : {Verdict(ecartRapport <= seuilRapportCite21)}."));
    Console.WriteLine(Inv($"  Coefficient : |R/prédiction − 1| = {DoubleFixe(ecartCoefficient):E6} ; seuil = {DoubleFixe(seuilCoefficient21):E1} : {Verdict(okCoefficient)}."));
    Verifier(ecartR <= tolR && ecartPred <= tolPred && ecartRapport <= seuilRapportCite21 && okCoefficient,
        Inv($"Contrôle du résidu ou du coefficient en défaut à Λ = 1e{cas.E}."));
}

// Verdict derive du ratio calcule, jamais d'une valeur figee : la loi de
// decroissance en 1/Lambda^2 impose R(1e2)/R(1e4) = (1e4/1e2)^2 = 1e4.
// L'ecart relatif mesure est compare a un seuil nomme, calibre sur la
// correction residuelle en 1/Lambda attendue (~1.5e-5). Attention : ce test
// ne voit que la LOI de decroissance (l'exposant 2) — un facteur commun se
// simplifie dans R(1e2)/R(1e4), donc un defaut de coefficient (du type
// pi^2/6 au lieu de pi^2/3) laisserait le verdict CONFORME. C'est la
// colonne 'rapport' (R/pred ~ 1 aux trois troncatures) qui valide le
// coefficient lui-meme.
// Plus précisément, la correction relative au terme dominant est en 1/Λ².
var attendu21 = BigInteger.Pow(10, 2 * (casResidu21[2].E - casResidu21[0].E)) * echelleFixe;
var ratio21 = DivFixe(ecartsResidu21[0].R, ecartsResidu21[2].R);
var rel21 = DivFixe(BigInteger.Abs(ratio21 - attendu21), attendu21);
var ecartRatio21 = EcartRefFixe(ratio21, RefRatio21);
var ecartRel21 = EcartRefFixe(rel21, RefRel21);
var ecartSeuil21 = EcartRefFixe(seuilExposant21, RefSeuil21);
bool okExposant21 = rel21 <= seuilExposant21;
string verdictExposant21 = okExposant21 ? "CONFORME" : "NON CONFORME";
Console.WriteLine();
AfficherComparaisonFixe("ratio R(1e2)/R(1e4) (cellule Python 21)", ratio21, RefRatio21, ecartRatio21, seuilRatioCite21);
AfficherComparaisonFixe("écart relatif à 1e4 (cellule Python 21)", rel21, RefRel21, ecartRel21, seuilRelCite21);
AfficherComparaisonFixe("seuil (cellule Python 21)", seuilExposant21, RefSeuil21, ecartSeuil21, BigInteger.Zero);
Console.WriteLine($"verdict = {verdictExposant21} ; Python, cellule 21, cité = {RefVerdict21} : {Verdict(verdictExposant21 == RefVerdict21)}.");
Verifier(okExposant21 && verdictExposant21 == RefVerdict21 && ecartRatio21 <= seuilRatioCite21
    && ecartRel21 <= seuilRelCite21 && ecartSeuil21.IsZero, "Le contrôle de la loi de décroissance a échoué.");

   Lam                 R(Lam)     pi^2 p^2/(3 Lam^2)      rapport


   1e2   0.000328981878661439  0.0003289868133696453 0.999985000286


  Python, cellule 21, cités dans l’ordre : 0.000328981878661439, 0.0003289868133696453, 0.999985000286.


  |ΔR| = 1.679438E-020 (seuil 1e-18) : CONFORME ; |Δ prédiction| = 1.270552E-020 (seuil 1e-19) : CONFORME ; |Δ rapport| = 2.916665E-013 (seuil 1.0E-012) : CONFORME.


  Coefficient : |R/prédiction − 1| = 1.499971E-005 ; seuil = 2.0E-005 : CONFORME.


   1e3  3.289867640216327E-06  3.289868133696453E-06   0.99999985


... (7 outputs left out)

ratio R(1e2)/R(1e4) (cellule Python 21) = 9999.850017856858332846293002944240656907254404711140834607149901 ; Python cité = 9999.85001785685833284495861 ; |Δ| = 1.334393E-018 ; seuil = 1.00E-016 : CONFORME


écart relatif à 1e4 (cellule Python 21) = 0.000014998214314166715370699705575934309274559528885916539285 ; Python cité = 0.0000149982143141667155041389997736 ; |Δ| = 1.334393E-022 ; seuil = 1.00E-020 : CONFORME


seuil (cellule Python 21) = 0.0005 ; Python cité = 0.0005 ; |Δ| = 0.000000E+000 ; seuil = 0.00E+000 : CONFORME


verdict = CONFORME ; Python, cellule 21, cité = CONFORME : CONFORME.


## 9. Exercices

Trois exercices, à compléter — le carnet s'exécute de bout en bout sans eux
(règle C.1).

In [11]:
// Illustration 1 — Exercice 1 : La bulle ne connaît p² que par le logarithme.
// Refaire le tableau de la section 4 avec p² = 4, puis vérifier que
// B(Λ; p²=4) − π²(ln(Λ²) − ln(4) + 1) tend vers 0.
// Indice : la prédiction vient de la forme fermée (remplacer Δ par 4 x(1−x)).
// Étape 1 : poser p²_exo = 4 et réévaluer la différence demandée à Λ = 10⁸.
BigInteger? resultatExo1 = null; // TODO étudiant
Console.WriteLine("Exercice a completer : dependance d'echelle de la bulle");

Exercice a completer : dependance d'echelle de la bulle


In [12]:
// Illustration 2 — Exercice 2 : La même bulle en d = 3.
// En trois dimensions la mesure est S² (aire 4*pi) et l’intégrande décroît
// comme 1/k^4 sur un volume en k^2 dk : le comptage de puissances prédit la
// convergence. Mesurer B3(Λ) à Λ = 10, 100 et vérifier que la suite se
// stabilise (contrairement au tableau de la section 4).
// Étape 1 : écrire B3Direct(Λ) sur le modèle de BDirect, avec la mesure
// 2*pi*sin(psi) dpsi et k^2 dk (volume S²).
double? resultatExo2 = null; // TODO étudiant
Console.WriteLine("Exercice a completer : bulle en d=3, convergence");

Exercice a completer : bulle en d=3, convergence


In [13]:
// Illustration 3 — Exercice 3 : Vérifier la forme radiale fermée par différences finies.
// La section 2 utilise integral_0^Λ k^3 dk/(k^2+Δ)^2 = 1/2[ln((Λ^2+Δ)/Δ)
// + Δ/(Λ^2+Δ) - 1]. Vérifier numériquement (quadrature directe contre formule)
// pour Δ = 0.1 et Λ = 3, à 25 chiffres.
// Étape 1 : comparer la quadrature de k^3/(k^2+0.1)^2 sur [0, 3] à la formule.
// Le double permet un premier contrôle, pas de certifier 25 chiffres :
// pour ce défi, les deux évaluations doivent avoir une précision suffisante.
double? resultatExo3 = null; // TODO étudiant
Console.WriteLine("Exercice a completer : forme radiale fermee");

Exercice a completer : forme radiale fermee


## 10. Ce que ce carnet établit

*Bilan du jumeau Python (`mpmath`) :*

- La **première divergence** de la série : la bulle en $d = 4$ croît comme
  $\pi^2 \ln \Lambda^2$, coefficient mesuré à $7 \times 10^{-14}$ (planted
  truth n° 1).
- La **constante finie $F = \pi^2$**, ancrée dans une intégrale calculable à
  la main ($\int_0^1 \ln 1/x(1-x)\,dx = 2$), mesurée à $3 \times 10^{-25}$
  par la forme fermée et à $3 \times 10^{-29}$ par la voie directe stabilisée —
  **sans résidu** : c'est la forme fermée qui en porte un,
  $\pi^2 p^2/(3\Lambda^2)$, terme de décalage et non erreur.
- La **régularisation soustractive** : $B_R(\mu^2) = \pi^2(1 + \ln\mu^2)$,
  limite finie exhibée, dépendance d'échelle mesurée — le premier « running »
  de la série.
- Trois **leçons de méthode** exposées, pas gommées : la mesure $S^3$ vérifiée
  avant usage (une première écriture rendait la moitié) ; l'écart
  $\bar\Delta/\Lambda^2$ de la boule dure face au décalage, mesuré et non
  masqué ; et la **quadrature appliquée sur un coin** ($k = p$), qui rendait
  une erreur constante de $2\times10^{-4}$ — diagnostiquée par un contrôle à
  quatre cases plutôt que rafistolée.

> **Bilan C# :** les sorties des §5 et §7 comparent à $\pi^2$ le coefficient logarithmique et la pente en $\ln\mu^2$. Le §6 distingue l'accord de la voie directe stabilisée au seuil absolu de $10^{-10}$ du résidu de la forme fermée, évalué à échelle fixe. L'amplitude de l'artefact naïf se lit dans les sorties des §3 et §6 : elle dépend du maillage et les valeurs de `mpmath` ne sont pas des mesures C#.

L'étage suivant du classement BootLoops (polylogarithmes → **elliptique** →
K3 → Calabi-Yau) reste ouvert : la structure de double période entrevue au
carnet 02 n'est pas développée ici. Le candidat 04 — s'il vit — la reprendra
depuis la courbe elliptique du sunrise.

## À retenir

*Les précisions chiffrées de ce tableau sont celles du jumeau Python (`mpmath`).*

| Notion | Ce qu'elle dit | Où la voir |
|---|---|---|
| Première divergence | $B(\Lambda) \sim \pi^2 \ln \Lambda^2$ — la bulle en $d = 4$ diverge logarithmiquement | §4, §5 |
| Coefficient mesuré | le coefficient de $\ln \Lambda^2$ est $\pi^2$, mesuré à $7 \times 10^{-14}$ (planted truth n° 1) | §5 |
| Constante finie | $F = \pi^2$, ancrée dans $\int_0^1 \ln\frac{1}{x(1-x)}\,dx = 2$, mesurée à $3 \times 10^{-25}$ (forme fermée) et $3 \times 10^{-29}$ (voie directe stabilisée) | §6 |
| Résidu, pas erreur | l'écart restant est $\pi^2 p^2 / (3\Lambda^2)$ — un terme de décalage que la forme fermée porte explicitement | §8 |
| Régularisation soustractive | $B_R(\mu^2) = \pi^2(1 + \ln\mu^2)$ : un paramètre, une échelle, une limite finie — le premier « running » de la série | §7 |

La mesure de l'aire de $S^3$ ($2\pi^2$, vérifiée avant usage) est le garde-fou qui a attrapé une première écriture qui rendait la moitié du facteur — les leçons de méthode du §10 sont partie du livrable.

> **Dans les sorties C# :** la différence entre deux maillages mesure la sensibilité au raffinement ; l'écart à la valeur analytique contrôle la précision de la quadrature (§1, §3, §6). L'erreur du témoin naïf à maillage fixé est un artefact (§3). Les rapports de résidus évalués à échelle fixe contrôlent le décalage analytique (§8).

## Vérifiez votre compréhension

*Les valeurs multiprécision dans les questions et réponses sont celles du jumeau Python (`mpmath`).*

1. La voie « naïve » porte un écart de $2{,}08 \times 10^{-4}$. Quel contrôle du §3 prouve que ce n'est **pas** un défaut de convergence, et qu'est-ce qui le répare ?

<details><summary>Réponse</summary>

La table du contrôle : l'écart naïf est **constant** ($2{,}08 \times 10^{-4}$ à $\Lambda = 2$, $8$ et $64$, chiffre pour chiffre) alors que la voie stabilisée converge ($0$, puis $6{,}3 \times 10^{-30}$, puis $1{,}3 \times 10^{-29}$). Un terme en $1/\Lambda^2$ décroîtrait ; un artefact constant, non. Le contrôle montre aussi que **changer de variable ne change rien** tant que le coin $k = p$ n'est pas séparé — la faute n'était donc pas dans l'intégrale angulaire où on la cherchait. Ce qui répare : séparer le coin dans la bonne variable ($t = \tan(\psi/2)$, où le $t^2$ du numérateur annule celui du dénominateur ; en $\psi$, la même séparation lève `DivideByZeroException` en C#).

En C#, relevez la dispersion entre coupures à maillage fixé et la variation sous raffinement. Le premier contrôle distingue l'artefact d'un terme en $1/\Lambda^2$ ; il ne prouve pas à lui seul la convergence quand le pas tend vers zéro. Le second contrôle, avec la comparaison à la formule analytique, teste la voie stabilisée.

</details>

2. Que mesure le planted truth n° 2, et pourquoi l'ancrage dans une intégrale calculable à la main rend-il l'identification robuste ?

<details><summary>Réponse</summary>

Que la constante finie $F$ vaut $\pi^2$. L'ancrage : $\int_0^1 \ln\frac{1}{x(1-x)}\,dx = 2$ se calcule à la main, et $F$ mesuré par la voie directe stabilisée ($\Lambda = 64$) redonne $\pi^2 = 9.8696044010893586188\ldots$ avec un écart de $3 \times 10^{-29}$. Reconnaître $\pi^2$ **après** avoir ancré la valeur dans une intégrale indépendante interdit la devinette rétroactive — la discipline *constant-recognition* à l'œuvre.

Dans le port, le §6 affiche séparément l'écart de la quadrature à $2$, celui de la partie finie directe à l'ancre et le résidu à échelle fixe de la forme fermée. Le test en `double` ne mesure pas un écart de $10^{-29}$.

</details>

3. En quoi le résidu de la forme fermée n'est-il **pas** une erreur de calcul ?

<details><summary>Réponse</summary>

Parce qu'il est **prédit par la forme fermée elle-même** et séparé de l'artefact par sa **loi d'échelle**, non par sa valeur : l'artefact est constant en $\Lambda$, le résidu $\pi^2 p^2/(3\Lambda^2)$ décroît. La table du §8 montre le rapport tendre vers 1 — c'est un terme de décalage analytique (l'impulsion externe $p$), que la voie directe stabilisée, elle, ne porte pas du tout.

Comparez dans le tableau C# le rapport à la prédiction, qui teste le coefficient, et le quotient entre coupures, qui teste l'exposant.

</details>

## Pour aller plus loin

- **Carnet suivant** : [Integrales-04 — Sunrise, courbe elliptique](Integrales-04-Sunrise-Courbe-Elliptique-Python.ipynb) — la structure de double période entrevue au carnet 02 et laissée ouverte ici est reprise depuis la courbe elliptique du sunrise : intégrale de première espèce, module $k^2(s)$, et la question de savoir si le candidat 04 vit.
- **Carnets voisins** : [Integrales-01 — Triangle scalaire](Integrales-01-Triangle-Scalaire-Python.ipynb) (l'ancre $I(0) = (4/\sqrt{3})\,Cl_2(\pi/3)$, protocole de reconnaissance des constantes) et [Integrales-02 — Sunrise, masses égales](Integrales-02-Sunrise-Masses-Egales-Python.ipynb) (le cas symétrique $m_1 = m_2 = m_3$ et son $\Lambda$-morfisme).

### Attestation de parité

Les valeurs de référence sont des citations de la sortie Python, jamais des résultats C#. Les écarts mesurés et les verdicts sont imprimés en regard des valeurs calculées dans les sorties ci-dessous ; un seuil déclaré ne constitue pas une mesure.

| Section | Écarts et verdicts à relever dans la sortie C# |
|---|---|
| §1 — Aire de $S^3$ | Écart absolu à la citation et à $2\pi^2$, écart relatif et variation sous raffinement. |
| §2 — Forme fermée | Pour $\Lambda=2,8$, écarts à la citation à échelle fixe et en quadrature, accord entre les deux évaluations et variation sous raffinement. |
| §3 — Voies directes | Écarts à la formule et aux citations pour les voies stabilisée et naïve, dispersion entre coupures à maillage fixé, exception au coin en $\psi$ et décroissance du décalage. Les lignes tronquées du tableau Python des deux voies ne sont pas citées comme si elles étaient disponibles. |
| §4–5 — Divergence | Écarts sur les parties finies et leurs rapports, puis sur le coefficient logarithmique, $\pi^2$ et leur écart relatif. |
| §6 — Constante finie | Écart de l'ancre à $2$ et variation sous raffinement ; écarts à la citation pour la forme fermée et pour chacune des deux voies directes. Un seuil manqué par le témoin naïf reste affiché. |
| §7 — Soustraction | Écarts à la citation pour chaque échelle, correction à coupure finie et écart de la pente à $\pi^2$. |
| §8 — Résidu | Écarts sur les résidus, les prédictions et les rapports ; verdicts distincts sur le coefficient et sur la décroissance en $1/\Lambda^2$. |

Les quadratures tanh–sinh en `double` sont comparées au seuil absolu déclaré de $10^{-10}$, sans assimilation à la précision de `mpmath`. Les formes fermées utilisent `BigInteger` à échelle $10^{60}$ ; cette échelle sert au calcul et n'est pas une garantie de précision des citations. La validation sur `.net-csharp` est établie par l'exécution intégrale et ses sorties enregistrées, non par cette note seule.

> Jumeau de référence : [Integrales-03 — La bulle et la première divergence (Python)](Integrales-03-Bulle-Premiere-Divergence-Python.ipynb).